# 03 — Train Regional Topic Models
Train a BERTopic model per National Park on globally-classified documents. Results are stored in DuckDB.

In [ ]:
import sys
sys.path.insert(0, '../src')

import yaml
from pathlib import Path

from reddit_np_topics.db import get_connection, read_table, write_dataframe, list_parks, table_row_count
from reddit_np_topics.modeling.train_regional import train_all_regional_models

with open('../config.yaml') as f:
    cfg = yaml.safe_load(f)

conn = get_connection(Path('..') / cfg['paths']['database'])

In [ ]:
# Load globally classified data
df = read_table(conn, 'global_classified')
parks = list_parks(conn, 'global_classified')
print(f'Loaded {len(df)} documents across {len(parks)} parks')
print(parks)

In [ ]:
# Optionally train a single park first to check output
# parks = ['YellowstoneNationalPark']

regional_df = train_all_regional_models(
    global_df=df,
    cfg=cfg['modeling'],
    models_dir=Path('..') / cfg['paths']['models_regional'],
    parks=parks,
)

In [ ]:
# Save to DuckDB
write_dataframe(conn, regional_df, 'regional_classified', mode='replace')
print(f'regional_classified: {table_row_count(conn, "regional_classified")} rows')